# Trace-Bench M2 -- Coverage + Parallel Execution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/guru-code-expert/Trace-Bench/blob/m2/coverage/notebooks/02_m2_coverage.ipynb)

This notebook validates **M2 contracts**:
- **Parallel execution** (max_workers > 1)
- **Resume/skip-existing** semantics
- **LLM4AD broad coverage** (65 tasks x 2 trainers)
- **Dynamic trainer resolution** (all OpenTrace trainers)
- **fail_fast** cancellation

**Mode policy**: defaults to **real** if API key present, else **stub**.

## Expected Outputs

- Parallel run produces same results as sequential (deterministic job IDs).
- Resume run skips completed jobs (manifest shows `status: reused`).
- LLM4AD coverage report: >=80% tasks load successfully.
- Coverage results.csv with all 65+ tasks.
- fail_fast stops early after first failure.

In [ ]:
# Mount Drive (optional) + compute persistent runs_dir + detect API key
from datetime import date
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass


def bench_dir(project="bench", sub="trace_bench_m2", local="/content/bench"):
    drive_root = Path("/content/drive/MyDrive")
    root = drive_root if drive_root.is_dir() else Path(local)
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)

RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)

# --- Auto-detect API key ---
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        pass

# Model: x-ai/grok-4.1-fast (primary) or deepseek/deepseek-v3.2 (cheaper)
MODEL = os.environ.get("OPENROUTER_MODEL", "openrouter/x-ai/grok-4.1-fast")

if API_KEY:
    os.environ["OPENROUTER_API_KEY"] = API_KEY
    os.environ["OPENAI_API_KEY"] = API_KEY
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    os.environ["TRACE_DEFAULT_LLM_BACKEND"] = "LiteLLM"
    os.environ["TRACE_LITELLM_MODEL"] = MODEL
    MODE = "real"
    print(f"API key found - running in REAL mode (model: {MODEL})")
else:
    MODE = "stub"
    print("WARNING: No OPENROUTER_API_KEY found. Falling back to STUB mode.")

os.environ["TB_MODE"] = MODE
print(f"\nMode: {MODE}")

In [ ]:
# Clone repos + install deps
!git clone --depth 1 --branch m2/coverage https://github.com/guru-code-expert/Trace-Bench.git 2>/dev/null || (cd Trace-Bench && git pull)
!git clone --depth 1 --branch experimental https://github.com/guru-code-expert/OpenTrace.git 2>/dev/null || (cd OpenTrace && git pull)

%cd Trace-Bench

!apt-get update -qq && apt-get install -y -qq graphviz > /dev/null 2>&1
!python -m pip install -q pyyaml pytest numpy matplotlib graphviz litellm==1.75.0 \
    "aiohttp>=3.9,<3.13" scipy networkx gymnasium gym pandas datasets sympy pymoo

## 1. Parallel Execution (max_workers > 1)

Run 2 tasks x 2 trainers in parallel (max_workers=2) and verify results match sequential.

In [ ]:
%%bash
set -euo pipefail
cd /content/Trace-Bench

echo "=== Parallel Execution Demo (max_workers=2) ==="

cat > /content/m2_parallel.yaml <<YAML
runs_dir: runs
mode: stub
seeds: [123]
max_workers: 2
fail_fast: false

tasks:
  - id: internal:numeric_param
  - id: internal:code_param

trainers:
  - id: PrioritySearch
    params_variants:
      - ps_steps: 1
        ps_batches: 1
  - id: GEPA-Base
    params_variants:
      - gepa_iters: 1
        gepa_train_bs: 2
YAML

echo "--- Sequential (max_workers=1) ---"
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config /content/m2_parallel.yaml --runs-dir "$RUNS_DIR/seq" --max-workers 1

echo ""
echo "--- Parallel (max_workers=2) ---"
PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config /content/m2_parallel.yaml --runs-dir "$RUNS_DIR/par" --max-workers 2

echo ""
echo "Both completed successfully."

In [ ]:
# Verify parallel produces same job_ids as sequential
import json, pathlib, pandas as pd

def latest_run(root):
    root = pathlib.Path(root)
    candidates = sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
    for p in reversed(candidates):
        if (p / "results.csv").exists():
            return p
    return None

seq_dir = latest_run(f"{RUNS_DIR}/seq")
par_dir = latest_run(f"{RUNS_DIR}/par")
print(f"Sequential run: {seq_dir}")
print(f"Parallel run:   {par_dir}")

df_seq = pd.read_csv(seq_dir / "results.csv")
df_par = pd.read_csv(par_dir / "results.csv")

seq_ids = sorted(df_seq["job_id"].tolist())
par_ids = sorted(df_par["job_id"].tolist())
assert seq_ids == par_ids, f"Job IDs differ!\nSeq: {seq_ids}\nPar: {par_ids}"
print(f"\nBoth produced {len(seq_ids)} jobs with matching IDs.")
print(f"Sequential: {len(df_seq)} rows")
print(f"Parallel:   {len(df_par)} rows")
df_par[["task_id", "trainer_id", "status", "score_best"]]

## 2. Resume / Skip-Existing Semantics

Re-run the same config -- completed jobs are reused, not re-executed.

In [ ]:
%%bash
set -euo pipefail
cd /content/Trace-Bench

echo "=== Resume Demo ==="
echo "Re-running the same config against $RUNS_DIR/par (should reuse all jobs)"

PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config /content/m2_parallel.yaml --runs-dir "$RUNS_DIR/par"

echo ""
echo "Done. Check manifest for 'reused' status."

In [ ]:
# Verify resume: manifest should show 'reused' for all jobs
import json, pathlib

par_root = pathlib.Path(f"{RUNS_DIR}/par")
runs = sorted([p for p in par_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
# The latest run should be the resumed one
resume_dir = runs[-1]
manifest = json.loads((resume_dir / "meta" / "manifest.json").read_text())

statuses = [j["status"] for j in manifest["jobs"]]
print(f"Resume run: {resume_dir.name}")
print(f"Job statuses: {statuses}")

reused_count = sum(1 for s in statuses if s == "reused")
print(f"\nReused: {reused_count}/{len(statuses)}")
if reused_count == len(statuses):
    print("All jobs reused -- resume works correctly!")
else:
    print(f"WARNING: {len(statuses) - reused_count} jobs were re-executed")

## 3. LLM4AD Broad Coverage

Attempt to load all 65 LLM4AD tasks and report coverage. Target: >=80% functional.

In [ ]:
import sys, os
sys.path.insert(0, "/content/OpenTrace")
sys.path.insert(0, "/content/Trace-Bench")

from trace_bench.registry import discover_llm4ad, load_task_bundle, ensure_llm4ad_importable
from pathlib import Path

TASKS_ROOT = Path("/content/Trace-Bench/LLM4AD/benchmark_tasks")
ensure_llm4ad_importable(TASKS_ROOT)
specs = discover_llm4ad(TASKS_ROOT)
print(f"Discovered {len(specs)} LLM4AD tasks\n")

results = {"ok": [], "failed": [], "skipped": []}
for spec in specs:
    try:
        bundle = load_task_bundle(spec.id, str(TASKS_ROOT))
        required = {"param", "guide", "train_dataset", "optimizer_kwargs", "metadata"}
        missing = required - set(bundle.keys())
        if missing:
            results["failed"].append((spec.id, f"missing: {sorted(missing)}"))
        else:
            results["ok"].append(spec.id)
    except NotImplementedError as exc:
        results["skipped"].append((spec.id, str(exc)))
    except Exception as exc:
        results["failed"].append((spec.id, str(exc)[:120]))

total = len(specs)
functional = len(results["ok"])
pct = functional / total * 100

print(f"Coverage: {functional}/{total} = {pct:.1f}%")
print(f"  OK:      {len(results['ok'])}")
print(f"  Failed:  {len(results['failed'])}")
print(f"  Skipped: {len(results['skipped'])}")

if pct >= 80:
    print(f"\nPASS: Coverage {pct:.1f}% >= 80% threshold")
else:
    print(f"\nBELOW TARGET: Coverage {pct:.1f}% < 80% threshold")

if results["failed"]:
    print("\nFailed tasks:")
    for tid, err in results["failed"]:
        print(f"  {tid}: {err}")

## 4. Full Coverage Run (Stub Mode)

Run all loadable LLM4AD tasks through BenchRunner with parallel execution.

In [ ]:
%%bash
set -euo pipefail
cd /content/Trace-Bench

echo "=== Full LLM4AD Coverage Run (mode=$TB_MODE, max_workers=4) ==="

PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run \
    --config configs/m2_coverage.yaml \
    --runs-dir "$RUNS_DIR/coverage" \
    --max-workers 4

echo ""
echo "Coverage run complete."

In [ ]:
# Analyze coverage results
import json, pathlib, pandas as pd

cov_root = pathlib.Path(f"{RUNS_DIR}/coverage")
runs = sorted([p for p in cov_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
cov_dir = runs[-1]
print(f"Coverage run: {cov_dir.name}")

df = pd.read_csv(cov_dir / "results.csv")
print(f"Total jobs: {len(df)}")
print(f"\nStatus breakdown:")
print(df["status"].value_counts())

ok_tasks = df[df["status"] == "ok"]["task_id"].nunique()
all_tasks = df["task_id"].nunique()
print(f"\nTasks with at least one OK: {ok_tasks}/{all_tasks} = {ok_tasks/all_tasks*100:.1f}%")

summary = json.loads((cov_dir / "summary.json").read_text())
print(f"\nSummary: {json.dumps(summary, indent=2)}")

In [ ]:
# Coverage table by task
import pandas as pd

task_status = df.groupby("task_id")["status"].apply(
    lambda x: "ok" if "ok" in x.values else x.iloc[0]
).reset_index()
task_status.columns = ["task_id", "best_status"]

print(f"\n{'Task':<65} {'Status'}")
print("-" * 80)
for _, row in task_status.sort_values("task_id").iterrows():
    marker = "OK" if row["best_status"] == "ok" else "FAIL" if row["best_status"] == "failed" else "SKIP"
    print(f"{row['task_id']:<65} [{marker}]")

## 5. fail_fast Demo

With `fail_fast: true`, execution stops after the first failure.

In [ ]:
%%bash
set -euo pipefail
cd /content/Trace-Bench

echo "=== fail_fast Demo ==="

cat > /content/m2_failfast.yaml <<YAML
runs_dir: runs
mode: stub
seeds: [123]
max_workers: 1
fail_fast: true

tasks:
  - id: internal:non_trainable
  - id: internal:numeric_param
  - id: internal:code_param

trainers:
  - id: PrioritySearch
    params_variants:
      - ps_steps: 1
YAML

PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run --config /content/m2_failfast.yaml --runs-dir "$RUNS_DIR/failfast"

echo ""
echo "Done -- check manifest for not_executed jobs."

In [ ]:
# Verify fail_fast: some jobs should be not_executed
import json, pathlib

ff_root = pathlib.Path(f"{RUNS_DIR}/failfast")
runs = sorted([p for p in ff_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
ff_dir = runs[-1]
manifest = json.loads((ff_dir / "meta" / "manifest.json").read_text())

print(f"fail_fast run: {ff_dir.name}")
print(f"Total jobs in manifest: {len(manifest['jobs'])}")
for j in manifest["jobs"]:
    print(f"  {j['task_id']:<35} {j['status']}")

executed = [j for j in manifest["jobs"] if j["status"] != "not_executed"]
not_executed = [j for j in manifest["jobs"] if j["status"] == "not_executed"]
print(f"\nExecuted: {len(executed)}, Not executed: {len(not_executed)}")
assert len(not_executed) >= 1, "fail_fast should prevent some jobs from running"
print("PASS: fail_fast correctly stopped execution after failure.")

## 6. Real-Mode Coverage (API Key Required)

Run the full coverage config in **real mode** using an actual LLM backend.
This cell only works when `OPENROUTER_API_KEY` is set.

Target: >=80% of tasks run successfully with real LLM calls.

In [ ]:
# Real-mode full coverage: dynamically discover ALL loadable LLM4AD tasks
import sys, os, json, tempfile

if os.environ.get("TB_MODE") != "real":
    print("SKIP: Real-mode requires OPENROUTER_API_KEY. Set it and re-run.")
else:
    sys.path.insert(0, "/content/OpenTrace")
    sys.path.insert(0, "/content/Trace-Bench")

    from trace_bench.registry import discover_llm4ad, ensure_llm4ad_importable
    from pathlib import Path

    TASKS_ROOT = Path("/content/Trace-Bench/LLM4AD/benchmark_tasks")
    ensure_llm4ad_importable(TASKS_ROOT)
    specs = discover_llm4ad(TASKS_ROOT)
    task_entries = [{"id": s.id} for s in specs]

    config = {
        "mode": "real",
        "seeds": [123],
        "max_workers": 4,
        "resume": "auto",
        "job_timeout": 600,
        "trainers": [
            {
                "id": "PrioritySearch",
                "params_variants": [
                    {
                        "ps_steps": 1,
                        "ps_batches": 1,
                        "ps_candidates": 2,
                        "ps_proposals": 2,
                    }
                ],
            }
        ],
        "tasks": task_entries,
    }

    config_path = "/content/m2_real_coverage_dynamic.json"
    Path(config_path).write_text(json.dumps(config, indent=2))
    print(f"Generated real-mode config with {len(task_entries)} tasks")
    print(f"Config written to: {config_path}")

    import subprocess
    env = dict(os.environ)
    env["PYTHONPATH"] = "/content/OpenTrace:" + env.get("PYTHONPATH", "")
    result = subprocess.run(
        [
            sys.executable, "-m", "trace_bench", "run",
            "--config", config_path,
            "--runs-dir", f"{os.environ['RUNS_DIR']}/real_coverage",
        ],
        cwd="/content/Trace-Bench",
        env=env,
        capture_output=True,
        text=True,
    )
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print(f"STDERR:\n{result.stderr[-2000:]}")
    print("\nReal-mode full coverage run complete.")

In [ ]:
# Analyze real-mode results (if available)
import json, pathlib, os

if os.environ.get("TB_MODE") != "real":
    print("SKIP: Real-mode not active. Set OPENROUTER_API_KEY to enable.")
else:
    import pandas as pd
    real_root = pathlib.Path(f"{RUNS_DIR}/real_coverage")
    if real_root.exists():
        runs = sorted([p for p in real_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
        if runs:
            real_dir = runs[-1]
            print(f"Real-mode run: {real_dir.name}")
            df_real = pd.read_csv(real_dir / "results.csv")
            print(f"Total jobs: {len(df_real)}")
            print(f"\nStatus breakdown:")
            print(df_real["status"].value_counts())

            ok_count = len(df_real[df_real["status"] == "ok"])
            total = len(df_real)
            pct = ok_count / total * 100
            print(f"\nReal-mode success rate: {ok_count}/{total} = {pct:.1f}%")

            if pct >= 80:
                print(f"PASS: Real-mode coverage {pct:.1f}% >= 80%")
            else:
                print(f"BELOW TARGET: Real-mode coverage {pct:.1f}% < 80%")

            # Per-task table
            print(f"\n{'Task':<65} {'Status':<8} {'Score'}")
            print("-" * 90)
            for _, row in df_real.sort_values("task_id").iterrows():
                marker = "OK" if row["status"] == "ok" else "FAIL" if row["status"] == "failed" else "SKIP"
                score = row.get("score_best", "N/A")
                print(f"  {row['task_id']:<63} [{marker}]  {score}")
        else:
            print("No real-mode runs found.")
    else:
        print("Real-mode output directory not found.")

## 7. Optimizing Subset (10 tasks, 10 steps)

Run 10 hand-picked tasks with 10 training steps in real mode.
Verify that `score_best > score_initial` for at least some tasks,
demonstrating actual optimization is happening.

In [ ]:
%%bash
set -euo pipefail
cd /content/Trace-Bench

if [ "$TB_MODE" != "real" ]; then
    echo "SKIP: Optimizing subset requires OPENROUTER_API_KEY. Set it and re-run."
    exit 0
fi

echo "=== Optimizing Subset (10 tasks, 10 steps, real mode) ==="

PYTHONPATH=/content/OpenTrace:$PYTHONPATH python -m trace_bench run \
    --config configs/m2_optimizing_subset.yaml \
    --runs-dir "$RUNS_DIR/optimizing"

echo ""
echo "Optimizing subset run complete."

In [ ]:
# Verify optimizing subset: score_best > score_initial for some tasks
import json, pathlib, os

if os.environ.get("TB_MODE") != "real":
    print("SKIP: Optimizing subset requires real mode.")
else:
    import pandas as pd
    opt_root = pathlib.Path(f"{RUNS_DIR}/optimizing")
    if opt_root.exists():
        runs = sorted([p for p in opt_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
        if runs:
            opt_dir = runs[-1]
            print(f"Optimizing run: {opt_dir.name}")
            df_opt = pd.read_csv(opt_dir / "results.csv")

            print(f"\n{'Task':<50} {'Initial':>10} {'Best':>10} {'Improved?':>10}")
            print("-" * 85)

            improved_count = 0
            for _, row in df_opt.iterrows():
                si = row.get("score_initial")
                sb = row.get("score_best")
                improved = ""
                if pd.notna(si) and pd.notna(sb):
                    if sb > si:
                        improved = "YES"
                        improved_count += 1
                    elif sb == si:
                        improved = "SAME"
                    else:
                        improved = "NO"
                print(f"  {row['task_id']:<48} {str(si):>10} {str(sb):>10} {improved:>10}")

            total_ok = len(df_opt[df_opt["status"] == "ok"])
            print(f"\nResults: {total_ok}/{len(df_opt)} tasks OK, {improved_count} showed improvement")
            if improved_count > 0:
                print("PASS: At least one task showed score_best > score_initial")
            else:
                print("NOTE: No tasks showed improvement (may need more steps or different model)")
        else:
            print("No optimizing runs found.")
    else:
        print("Optimizing output directory not found.")

## Summary

M2 validation complete:
- **Parallel execution** produces deterministic, matching results.
- **Resume semantics** (auto/failed/none) skip completed jobs on re-run.
- **LLM4AD coverage** report shows task loading health.
- **Full coverage run** exercises all tasks through the runner.
- **fail_fast** correctly cancels pending jobs after failure.
- **Real-mode coverage** (with API key) validates actual LLM integration.
- **Optimizing subset** (10 tasks, 10 steps) demonstrates score improvement.